# $B^+\to K^+\pi^+\pi^-$ toy generation and signal-only fit

Generate and fit a non-CP $B^+\to K^+\pi^+\pi^-$ isobar toy using Square-Dalitz normalization. The Dalitz coordinates shown below are $s_{13}=m^2(K^+\pi^-)$ and $s_{23}=m^2(\pi^+\pi^-)$.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    BaBarFlatte, DecayChannel, DecayModel, LASS, Minimizer, NonResonant,
    Parameter, PhaseSpaceSample, RealImag, Resonance, enable_x64,
    weighted_resample,
)
from dalitzplotfitter.background import FunctionalBackground
from dalitzplotfitter.efficiency import FunctionalEfficiency

enable_x64()


## 1. Amplitude model and normalization


In [ ]:
channel = DecayChannel("B+", ("K+", "pi+", "pi-"))

truth_xy = {
    "Kstar892": (1.00, 0.00),
    "KpiS": (1.40, -0.60),
    "rho770": (0.65, 0.10),
    "f0_980": (-0.20, 1.00),
    "NR": (-0.50, 0.10),
}
truth = {}
def coefficient(name, fixed=False):
    x, y = truth_xy[name]
    if fixed:
        return RealImag(x, y)
    truth[f"{name}.x"], truth[f"{name}.y"] = x, y
    return RealImag(
        Parameter.coefficient(f"{name}.x", x, owner=name, step=0.01),
        Parameter.coefficient(f"{name}.y", y, owner=name, step=0.01),
    )

c = {name: coefficient(name, fixed=(name == "Kstar892")) for name in truth_xy}
components = [
    Resonance("Kstar892", (0,2), c["Kstar892"], mass=0.8958, width=0.0474, spin=1, resonance_radius=4.0, parent_radius=4.0),
    Resonance("KpiS", (0,2), c["KpiS"], lineshape=LASS(2.07, 3.32, 1.8), mass=1.425, width=0.270, spin=0, resonance_radius=4.0, parent_radius=4.0),
    Resonance("rho770", (1,2), c["rho770"], mass=0.7753, width=0.1491, spin=1, resonance_radius=4.0, parent_radius=4.0),
    Resonance("f0_980", (1,2), c["f0_980"], lineshape=BaBarFlatte(), mass=0.965, width=0.0, spin=0, resonance_radius=4.0, parent_radius=4.0),
    NonResonant(c["NR"]),
]
model = DecayModel(
    channel, components,
    normalization_method="square-dalitz",
    normalization_resolution=350,
    normalization_pair=(0, 2),
)
norm = model.normalization_sample
print("free parameters:", len(model.parameters))
print("normalization points:", norm.size)
model.print_fit_fractions(truth, normalization_sample=norm, include_interference=True)


## 2. Generate signal pseudo-data


In [ ]:
N_POOL = 200_000
N_DATA = 30_000
pool = model.generate_phase_space(N_POOL, seed=2000)
pool_cache = model.prepare_cache(pool, norm)
target_weights = pool.weights * pool_cache.intensity(truth)
data = weighted_resample(
    jax.random.key(791), pool, target_weights, N_DATA, replace=True
)
print("generated events:", data.size)

fig, ax = plt.subplots(figsize=(7, 5.5))
h = ax.hist2d(np.asarray(data.s13), np.asarray(data.s23), bins=90)
fig.colorbar(h[3], ax=ax, label="events")
ax.set(xlabel=r"$s_{13}$ [GeV$^2$]", ylabel=r"$s_{23}$ [GeV$^2$]")
plt.show()


## 3. Unbinned fit and closure table


In [ ]:
cache = model.prepare_cache(data, norm)
def nll(values):
    intensity, normalization = cache.evaluate(values)
    return -jnp.sum(jnp.log(jnp.clip(intensity, min=1e-300))) + data.size*jnp.log(normalization)

rng = np.random.default_rng(314159)
start = {
    parameter.name: truth[parameter.name] + rng.normal(0.0, 0.15)
    for parameter in model.parameters if not parameter.fixed
}
result = Minimizer(nll, model.parameters, verbose=1).fit(
    start_values=start, simplex=True, ncall=30_000
)
fit_values = {
    parameter.name: float(result.values[parameter.name])
    for parameter in model.parameters if not parameter.fixed
}
print("valid:", result.valid, "NLL:", result.fval, "EDM:", result.fmin.edm)
print(f"{'parameter':18s} {'generated':>11s} {'fitted':>11s} {'error':>11s} {'pull':>9s}")
for parameter in model.parameters:
    if parameter.fixed:
        continue
    name = parameter.name
    fitted = float(result.values[name]); error = float(result.errors[name])
    pull = (fitted-truth[name])/error
    print(f"{name:18s} {truth[name]:11.5f} {fitted:11.5f} {error:11.5f} {pull:9.3f}")

model.print_fit_fractions(
    fit_values, normalization_sample=norm, include_interference=True
)


## Fit projections

Compare the toy data with the injected model and the fitted model.


In [ ]:
projection_cache = model.prepare_cache(pool, norm)

def signal_projection(values, variable, bins):
    intensity, normalization = projection_cache.evaluate(values)
    weights = np.asarray(pool.weights * intensity / normalization)
    histogram, _ = np.histogram(
        np.asarray(getattr(pool, variable)), bins=bins, weights=weights
    )
    return histogram

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
for axis, variable, label in zip(
    axes,
    ("s13", "s23"),
    (r"$s_{13}$ [GeV$^2$]", r"$s_{23}$ [GeV$^2$]"),
):
    observed = np.asarray(getattr(data, variable))
    bins = np.linspace(observed.min(), observed.max(), 70)
    centers = 0.5*(bins[:-1] + bins[1:])
    data_hist, _ = np.histogram(observed, bins=bins)
    generated_hist = signal_projection(truth, variable, bins)
    fitted_hist = signal_projection(fit_values, variable, bins)
    generated_hist *= data_hist.sum()/generated_hist.sum()
    fitted_hist *= data_hist.sum()/fitted_hist.sum()

    axis.errorbar(
        centers, data_hist, yerr=np.sqrt(np.maximum(data_hist, 1.0)),
        fmt=".", color="black", label="toy data",
    )
    axis.step(centers, generated_hist, where="mid", linestyle="--", label="generated model")
    axis.step(centers, fitted_hist, where="mid", label="fitted model")
    axis.set(xlabel=label, ylabel="events / bin")
    axis.legend()
plt.show()


## Interpretation

A single pseudoexperiment is a workflow and closure demonstration. Bias and coverage require an ensemble of statistically independent toys.
